# 02 - Rank Momentum

This notebook computes Clenow-style momentum rankings from the cached S&P 500 OHLCV data. The default score is annualized exponential regression slope multiplied by R-squared.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from clenow.data import load_prices
from clenow.ranking import RankingConfig, rank_momentum

In [ ]:
PRICES_PATH = PROJECT_ROOT / "data" / "processed" / "sp500_prices.parquet"
RANKINGS_PATH = PROJECT_ROOT / "data" / "processed" / "momentum_rankings.parquet"
RANKINGS_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "momentum_rankings.csv"

config = RankingConfig(
    lookback_days=90,
    atr_window=20,
    trend_ma_window=100,
    trading_days=252,
    require_positive_slope=True,
    require_above_trend_ma=True,
    min_price=5.0,
    min_avg_dollar_volume=10_000_000.0,
)
config

In [ ]:
prices = load_prices(PRICES_PATH)
print(f"Loaded {len(prices):,} rows across {prices['ticker'].nunique()} tickers")
print(f"Date range: {prices['date'].min().date()} to {prices['date'].max().date()}")
prices.head()

In [ ]:
rankings = rank_momentum(prices, config=config)
RANKINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
rankings.to_parquet(RANKINGS_PATH, index=False)
rankings.to_csv(RANKINGS_CSV_PATH, index=False)

print(f"Ranked {len(rankings)} eligible stocks")
rankings.head(25)

In [ ]:
columns = [
    "rank",
    "ticker",
    "date",
    "last_price",
    "momentum_score",
    "annualized_slope",
    "r_squared",
    "ATR20",
    "MA100",
    "avg_dollar_volume_20",
]
rankings[columns].head(50)

In [ ]:
ax = rankings.head(30).plot.bar(
    x="ticker",
    y="momentum_score",
    figsize=(14, 5),
    title="Top 30 Clenow Momentum Scores",
    legend=False,
)
ax.set_ylabel("Annualized slope x R-squared");